# Imports and Globals

## Imports
 1. ```from __future__ import annotations``` 
    - Changes how type hints are handled, treats type annotations more lazily and cleanly
    - Handles things like (count: int = 4 or name: String = "Adam")
    - Normally, type hints are evaluated immediately. With this import, Python stores them in a more delayed/lazy form.
      - It can make typing cleaner and avoid some problems with forward references.
 2. ```import argparse```
    - Used to read and handle Command Line Aruguments(ex. item name, output filename, currency, country, language)
    - Essentially just imports Python's Command Line Parser, allowing for the use of:
         - Ex. Typing "Ak 47 | Safari Mesh (Minimal Wear) -o safari_mesh_listing.xlsx -USA -English ... <- this is using argparse, the script will understand this only through this import
 3. ```import time```
    - Used for time.sleep() so the script pauses between requests and does not spam Steam or the float API
 4. ```from dataclasses import dataclass```
    - Used to make an object for each listing row i.e allows for the use of @dataclass
 5. ```from typing import Any, Dict, Iterable, List, Optional```
    - Type hints to make the code clearer --> Type Hint ex. number: int = 5
    - These are type hints. They help describe what kinds of values a variable or function is expected to use.
         - `Any`: Means the value can be any type.
         - `Dict`: Means dictionary type.
         - `Iterable`: Means something you can iterate over, like a generator or list.
         - `List`: Means list type.
         - `Optional`: Means a value can either be: the given type or `None`
 6. ```from urllib.parse import quote```
    - Used to URL-encode the markey hash name so it can safely go into a Steam URL
    - Ex. Turns "AK 47 | Safari Mesh (Minimal Wear)" into a Steam URL to the community listing
 7. ```import pandas as pd```
    - Import that pandas class to create a dataframe and import to excel
 8. ```import requests```
    - Used to to make HTTP Web requests to:
        - Steam
        - CSGOFloat API

In [ ]:
from __future__ import annotations #Changes how type hints are handled, treats type annotations more lazily and cleanly

import argparse #Used to read Command Line Aruguments(ex. item name, output filename, currency, country, language)
import time #Used for time.sleep() so the script pauses between requests and does not spam Steam or the float API
from dataclasses import dataclass #Used to make an object for each listing row i.e allows for the use of @dataclass
from typing import Any, Dict, Iterable, List, Optional #Type hints to make the code clearer --> Type Hint ex. number: int = 5
from urllib.parse import quote #Used to URL-encode the market hash name so it can safely go into a Steam URL.

import pandas as pd
import requests

STEAM_APP_ID = 730
STEAM_CONTEXT_ID = 2
PAGE_SIZE = 10


## Global Constants
- Steam app ID: 730 is the Steam APP ID for CS2/CS:GO Market Items
- Steam Context ID = Identifies the Steam Invetory Context for the game's item inventory
- Page Size: Assumes the total number of listings on page of the Market to be 10
- After fetching a page request, the script waits 1 second before requesting the next page, is also turned into a command line argument for more user flexibility
- After each float API lookup the script waits 0.3 secods to prevent it from hammering API with thousands of float API calls
- If steam limits the script with HTTP 429, the script will retry up to 5 times.

In [ ]:
STEAM_APP_ID = 730
STEAM_CONTEXT_ID = 2
PAGE_SIZE = 10
DEFAULT_STEAM_PAGE_DELAY = 1.0
DEFAULT_FLOAT_API_DELAY = 0.3
DEFAULT_STEAM_RETRIES = 5

# Helper Functions

## Object Creation
- Creates a class that will be used to create class for the each listing, i.e each listing will be an object of the ListingRow class
- __init__ dunder method not needed as we use @dataclasses to remove boilerplate code
- ```Optional[type]``` = means that the value will either be ```type``` or ```none```

In [ ]:
@dataclass
class ListingRow:
    listing_id: str
    asset_id: str
    page: int
    price: float
    currency: str
    float_value: Optional[float]
    wear: Optional[str]
    paint_seed: Optional[int]
    has_stickers: Optional[bool]
    sticker_count: Optional[int]
    inspect_link: Optional[str]


## Turn Float into Wear
- Turns the float value of the skin listing, into a wear that falls into 5 catagorys: Factory New, Minimal Wear, Field-Tested, Well-worn, Battle-Scarred
- The portion ```(float_value: Optional[float]) -> Optional[str]``` simply says that the parameter for this function will either be a ```float``` or ```none```, and the return for the function will either be a ```str``` or ```none```

In [ ]:
def get_wear_from_float(float_value: Optional[float]) -> Optional[str]:
    if float_value is None:
        return None
    if float_value < 0.07:
        return "Factory New"
    if float_value < 0.15:
        return "Minimal Wear"
    if float_value < 0.38:
        return "Field-Tested"
    if float_value < 0.45:
        return "Well-Worn"
    return "Battle-Scarred"

## Normalize Inspect Link
- The inspect link is the specilized URL in the Steam Market, that allows plaayers to view the skin in game, and its further details, including the float, pattern index, and applied stickers
- The above function takes that raw link(given as a string), the listing id(), and the asset id, and then if the raw link contains either ```"%listingid%"``` or `"%assetid%`, it replaces them with the values for the actual IDs.
- This is important for generating an actually usable inspect link which allows us to get the other specifics of the listing

In [ ]:
def normalize_inspect_link(raw_link: str, listing_id: str, asset_id: str) -> str:
    return raw_link.replace("%listingid%", listing_id).replace("%assetid%", asset_id)


## Steam Render Function
- Essentially, the funciton asks Steam for one page of listing data in JSON format
- Instead of scrapping HTML, it uses Steam's structured market endpoint
    - This is much better than scraping HTML because JSON is:
        - cleaner
        - more structured
        - easier to loop over
        - easier to debug
### Step-by-Step
1. `session: requestions.Session()`
    - This is the resuable HTTP session the script uses for web requests
    - A similar way to do this would be to use `requests.get()` everytime
        - However a creating a Session object maintains "state" (memory of previous actions) across multiple interactions and it also:
            - Resuses connections
            - Keeps headers together
            - Is just more efficient when making many requests
2. `market_hash_name: str`
    - This is the Steam Market item name, ex. `"Ak 47 | Safari Mesh(Minimal Wear)" `
3. `start: int `
    - This is the offset into the result set (Ex. `0` -> start at listing 1; `10` -> start at listing 10...)
4. `currency: int` / `country: str`/ `langauge: str`
    - Other properties of the listing that need to be determined
5. `mas_retries: int = DEFUALT_STEAM_RETRIES`
    - Maximum number of retries if Steam returns HTTP 429
6. ` -> Dict[str,Any]`
    - Return type of the function. The dictionary contains key value pairs such that the keys are all strings, but the pairs can be Any type(bool, int, list, etc.)
7. `encoded_name = quote(market_hash_name, safe="")`
    - Turns the user given name of the item into a URL-safe string
    - safe="" tells the quote function not leave special characters unescaped, i.e turn everything into the URL-safe string, don't preserve any of the original item name
        - Ex. `quote("/users/name/")` becomes `%2Fusers%2Fname%2F`; but Using `safe`: `quote("/users/name/", safe="/")` becomes `/users/name/`
        - By default, only letters, digits, and _.-~ are safe
8. `url = f"https://steamcommunity.com/market/listings/{STEAM_APP_ID}/{encoded_name}/render/"`
    - This builds the Steam Endpoint URL
    - The STEAM_APP_ID is inserted(730)
    - The encoded name from the prevoius line inserted
9. ##### `params...`
    - These are simply the URL query parameters for the request. 
    - Essentially the URL variables that were initialized at the beginning of the function are assigned to these query parameters
10. `attempt = 0`
    - Tracks how many retry attempts have happened so far.
    - Initial value is 0 because no retries have happened so far
11. `while True`
    - Infinite loop
    - Keeps trying until return suceeds
12. `response = session.get(url, params=params, timeout=25)`
    - This sends the GET request to Steam
    - It uses the URL created above for the item
    - Appends the query paramters
    - If steam takes too long, it will fail after 25 seconds
13. `if response.status_cpde != 429`
    - Essentially just checks for a HTTP 429 Error, if the error exist, it goes to retry logic, otherwise it follows the if statement
        - 13a. Normal Sucess
            - `response.raise_for_status()` --> if the HTTP response is bad raise an excpetion
            - `payload = response.json()` --> Parse the JSON response into python data
            - `if not payload.get("sucesss", False)` --> Steam's JSON usually contains a sucess flag, if it is false or missing, the conditional raises an error
            - `return payload` --> If everything is okay, return the JSON payload
        - 13b. Retry Logic
            - `attempt +=1` --> If the code reaches here, a HTTP 429 was recieved, so we increase the retry attempt number
            - `if attempt > max_retries` --> If the number of retries is greater than the allowed max, we stop and raise an HTTP error
                - `response=response` --> attaches the original response to the excpetion, which can help with debugging
            - 

In [ ]:
def steam_render_page(
    session: requests.Session,
    market_hash_name: str,
    start: int,
    currency: int,
    country: str,
    language: str,
    max_retries: int = DEFAULT_STEAM_RETRIES,
) -> Dict[str, Any]:
    encoded_name = quote(market_hash_name, safe="")
    url = f"https://steamcommunity.com/market/listings/{STEAM_APP_ID}/{encoded_name}/render/"
    params = {
        "start": start,
        "count": PAGE_SIZE,
        "currency": currency,
        "language": language,
        "country": country,
        "format": "json",
    }
    attempt = 0

    while True:
        response = session.get(url, params=params, timeout=25)
        if response.status_code != 429:
            response.raise_for_status()
            payload = response.json()
            if not payload.get("success", False):
                raise RuntimeError(
                    f"Steam render endpoint returned unsuccessful response for start={start}"
                )
            return payload

        attempt += 1
        if attempt > max_retries:
            raise requests.HTTPError(
                f"Steam rate limited the render endpoint after {max_retries} retries for start={start}",
                response=response,
            )

        retry_after = response.headers.get("Retry-After")
        try:
            wait_seconds = float(retry_after) if retry_after is not None else 0.0
        except ValueError:
            wait_seconds = 0.0

        wait_seconds = max(wait_seconds, min(5 * attempt, 30))
        print(
            f"Steam rate limited page starting at {start}. "
            f"Waiting {wait_seconds:.1f}s before retry {attempt}/{max_retries}..."
        )
        time.sleep(wait_seconds)

